In [1]:
import pandas as pandas
import geopandas as gpd
import numpy as np
import os

In [49]:
# capa links OSMNX con las velocidades y capacidades de TransCAD que se pudo traer Daniel en 01_velocidades_fromTransCAD
folder = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/VisumLinks With TransCAD Atts'
osmnx_links_con_atributos_01 = gpd.read_file(os.path.join(folder, "edges_osnmx_con_velocidades_y_capacidad2.shp"))

## Limpieza de atributos heredados incorrectamente

A partir del resultado de `osmnx_links_con_atributos_01` (`edges_osnmx_con_velocidades_y_capacidad2.shp (Daniel) merged con edges_osnmx_con_velocidades_y_capacidad2.shp (Jeannette)`), algunos links de OSMNX heredaron atributos por cercanía aunque no debían hacerlo.  
Esto ocurrió en elementos como ciclovías, líneas de tren, caminos peatonales y otros tipos de vía que no corresponden a la red vial principal.

### Eliminación de capacidad y velocidad en links no válidos

Para evitar asignaciones incorrectas, se eliminan los atributos de capacidad y velocidad en todos los links que no debían recibirlos.

In [50]:
#Cols
nombre = "name"
capacidad = "CAPACIDAD"
vel_prom = "Velocidad_"
v0 = "Limite_vel"

# Vias que no deben tener atributo capacidad ni velocidad (si lo tienen borrar!)
# Estos tipos de vias heredaron accidentalmente capacidades y velocidades de las avenidas principales solo por estar en ella o cercana a ella
highways_invalidos = [
    "residential",      #calles residenciales
    "living_street",    #calles residenciales
    "cycleway",         #ciclovias
    "footway",          #pasos peatonales
    "pedestrian",       #pasos peatonales
    "path",             #pasos peatonales
    "steps",            #pasos peatonales
    "busway",           #carriles de BRT
    "service",          #cuchillas/entradas/salidas estan catolgadas bajo este tag (ej. una cuchilla en osmnx links no debe tener la cap. y vel. de una avenida de transcad)
    "primary_link",     #cuchillas/entradas/salidas estan catolgadas bajo este tag
    "trunk_link",       #cuchillas/entradas/salidas estan catolgadas bajo este tag
    "motorway_link",    #cuchillas/entradas/salidas estan catolgadas bajo este tag
    "secondary_link",   #cuchillas/entradas/salidas estan catolgadas bajo este tag
    "tertiary_link"     #cuchillas/entradas/salidas estan catolgadas bajo este tag
]

# si es linea de tren ligero o si su columna highway esta en los invalidos, le vamos a borrar lo que heredo en 01_velocidades_fromTransCAD
mask_invalidos = (
    osmnx_links_con_atributos_01["highway"].isna() | #if highway = NULL es linea del metro
    osmnx_links_con_atributos_01['highway']
    .astype(str)
    .str.contains("|".join(highways_invalidos), case=False, na=False)
)

print("Links con atributos antes de limpiar (incluyendo highways invalidos):")
print(
    (
        osmnx_links_con_atributos_01[capacidad].notna() &
        osmnx_links_con_atributos_01[vel_prom].notna() &
        osmnx_links_con_atributos_01[v0].notna() 
    ).sum()
)

print("Links a los que les vamos a borrar lo que heredaron porque por su tipo de highway no debieron heredar en 1er lugar:")
print(
    (
        mask_invalidos &
        osmnx_links_con_atributos_01[capacidad].notna() &
        osmnx_links_con_atributos_01[vel_prom].notna() &
        osmnx_links_con_atributos_01[v0].notna() 
    ).sum()
)

# 1) Change to NaN the attributes of links which highway type shouldn't have gotten attributes
# Ex. Living streets or cycleways that got the attributes of a main avenue (wrong!)
osmnx_links_con_atributos_01.loc[mask_invalidos, [capacidad, vel_prom, v0]] = np.nan

# Total links con atributos heredados (after cleaning invalid highways' attributes)
links_con_atributos = osmnx_links_con_atributos_01[
    osmnx_links_con_atributos_01[capacidad].notna() &
    osmnx_links_con_atributos_01[vel_prom].notna() &
    osmnx_links_con_atributos_01[v0].notna() 
].copy()
print("Links con atributos despues de limpiar:")
print(len(links_con_atributos))

Links con atributos antes de limpiar (incluyendo highways invalidos):
12083
Links a los que les vamos a borrar lo que heredaron porque por su tipo de highway no debieron heredar en 1er lugar:
4863
Links con atributos despues de limpiar:
7220


In [51]:
# Todos estos tipos de highway es correcto que carguen atributo de TransCAD (ej. las invalid no lo debian traer att.)
links_con_atributos['highway'].value_counts()

highway
primary                 2875
secondary               2097
tertiary                1216
trunk                    569
unclassified             265
motorway                 187
track                     10
['primary', 'trunk']       1
Name: count, dtype: int64

### Save cleaned links con atributos heredados
- las ciclovias, pasos peatonales, lineas de mtro, etc ya no tendran atributos heredados

In [52]:
# Save all links (osmnx_links_con_atributos_01 ya le cambiamos a nan donde no debian tener atts. de capacidad y vel)
out_subfolder = "OSM Links con TransCAD Atts 01 Cleaned"
osmnx_links_con_atributos_01.to_file(os.path.join(folder, out_subfolder, "edges_osmnx_con_vel_y_cap_cleaned.shp"))